# GFlowNet Experiment — Kaggle/Colab Runner

**One-time setup:**
1. Enable GPU accelerator (Kaggle: *Settings → Accelerator → GPU T4 x2*; Colab: *Runtime → Change runtime type → T4 GPU*)
2. Add your GitHub token as a secret named `GITHUB_TOKEN` (needs `repo` scope)
   - Kaggle: *Add-ons → Secrets → Add*
   - Colab: click the 🔑 icon in the left sidebar → *Add new secret*

Then run all cells top to bottom.

In [ ]:
# 1. Install dependencies (numpy<2.0 required by mavenn/scipy)
# We do NOT restart the kernel — instead the experiment runs in a subprocess
# so a fresh Python interpreter picks up the downgraded numpy.
import subprocess, sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'numpy<2.0',
     'scipy>=1.11,<1.14',
     'scikit-learn',
     'pandas',
     'mavenn',
    ],
    check=True,
)
print('Dependencies installed.')

In [ ]:
# 2. Verify GPU (run after kernel restart)
import torch
import numpy as np
print('numpy:', np.__version__)
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — enable GPU accelerator in settings')

In [ ]:
# 3. Clone private repo
# Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    GITHUB_TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception:
    # Colab secrets fallback
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

REPO = 'YOUR_GITHUB_USERNAME/YOUR_REPO_NAME'  # <-- update this once
CLONE_DIR = '/kaggle/working/thesis'

import subprocess, sys, os
subprocess.run(['git', 'clone', f'https://{GITHUB_TOKEN}@github.com/{REPO}.git', CLONE_DIR], check=True)

sys.path.insert(0, CLONE_DIR)
os.chdir(CLONE_DIR)
os.makedirs('results', exist_ok=True)
print('Repo cloned. Working directory:', os.getcwd())

In [ ]:
# 4. Run experiment
# Change EXPERIMENT to 'gfn_tfbind8' or 'gfn_gb1'
EXPERIMENT = 'gfn_gb1'

from experiment_runner import ExperimentRunner
runner = ExperimentRunner(keyword=EXPERIMENT)
df = runner.run_and_save(parallel=False)  # parallel=False: single GPU, no multiprocessing

In [ ]:
# 5. Copy results to Kaggle output folder (auto-saved, downloadable from the notebook page)
import shutil, glob, os
OUTPUT_DIR = '/kaggle/working/results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

for f in glob.glob('results/*.csv'):
    dest = os.path.join(OUTPUT_DIR, os.path.basename(f))
    shutil.copy(f, dest)
    print(f'Saved: {f} → {dest}')

print('Done — download from the Kaggle notebook Output tab on the right.')